# 3. Feature Engineering (Ingeniería de Características)

**Punto de partida:** el dataset crudo (`../data/WA_Fn-UseC_-Telco-Customer-Churn.csv`), el mismo archivo usado en `2_EDA.ipynb`. Este notebook se ejecuta de forma independiente y vuelve a cargar los datos desde cero (no depende de variables en memoria de otros notebooks), de manera que pueda ejecutarse por separado como módulo de clase.

La ingeniería de características es la etapa del proceso analítico en la que transformamos, combinamos o construimos variables con el objetivo de representar mejor el fenómeno de negocio y preparar los datos para el modelamiento analítico.

Los datos dejan de ser únicamente registros operativos y se convierten en variables analíticas informativas, interpretables y reproducibles.

**Entrega de este notebook:** `../data/preprocessed_data.csv`, el archivo que consume directamente `4_Modeling.ipynb`.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

## cargar librerias de visualizacion
import seaborn as sns
import matplotlib.pyplot as plt

## Cargar datos

Se lee nuevamente el dataset crudo. A diferencia de `2_EDA.ipynb`, aquí sí se especifican explícitamente `sep` y `na_values`, ya que las decisiones de limpieza tomadas en el EDA (por ejemplo, qué se considera un valor faltante) se aplican de forma explícita desde la carga.

In [ ]:
## Cargar datos
fName = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

raw = pd.read_csv(
                    fName
                    , sep=",", na_values=[""," ","-","NA"]
                )
print("Dimensiones del dataset original:", raw.shape)

Se crea una copia del `DataFrame` original (`raw`) sobre la cual se trabajará (`df`), se define el `target` y se establece `customerID` como índice — la misma decisión validada en el EDA.

In [ ]:
df = raw.copy()
target = "Churn"
df.set_index("customerID", inplace=True)
print("Dimensiones de `df`:", df.shape)
df.head()

### Distribución del target

Se revisa nuevamente el balance de clases de `Churn`. Este dato es clave más adelante, en `4_Modeling.ipynb`, para decidir usar partición estratificada y priorizar métricas como Recall o F1 sobre Accuracy.

In [ ]:
pd.concat([df[target].value_counts(), (df[target].value_counts(normalize=True)*100).round(2)], axis=1)

## Preparación del dataset para implementación de ML

A partir de aquí se implementan las revisiones y técnicas de preparación necesarias para dejar el dataset listo para modelado, teniendo en cuenta cada característica, su distribución y su tipo de dato. Las decisiones que siguen retoman directamente los hallazgos del EDA (`2_EDA.ipynb`).

### Validación de nulos

In [ ]:
## Verificar valores nulos
df.isnull().sum()[df.isnull().sum()>0]

In [ ]:
nulos = df[df.isnull().sum(axis=1)>0]
print("Registros que tiene al menos un valor NULO =", len(nulos))
nulos

In [ ]:
print("Distribución de la variable objetivo en los registros con al menos un valor NULO")
pd.concat([nulos[target].value_counts(), nulos[target].value_counts(normalize=True)], axis=1)

### ¿Qué hacer con los valores nulos?

Las técnicas o procesos más comunes para tratarlos son:
* Eliminación de los registros.
* Imputación por promedio/mediana.
* Evaluación del contexto de negocio (¿por qué está nulo este dato específicamente?).

In [ ]:
# Descripción estadística de la variable `TotalCharges`
df[["TotalCharges"]].describe().T

In [ ]:
sns.displot(df["TotalCharges"], kde=True, height=4, aspect=2)
plt.title("Distribución de la variable `TotalCharges`")
plt.xlabel(None)
plt.ylabel(None)
plt.show()

In [ ]:
nulos.describe().T

De acuerdo con la distribución estadística de la característica y las técnicas disponibles para manejar nulos, se pueden implementar cualquiera de estas estrategias:

1. Descartar los registros, ya que corresponden solo a 11 (0.15%) registros del total.
2. Imputar por el promedio, que corresponde a `2283.3` — *en este contexto, y de acuerdo con la    distribución, no es lo más acertado*.
3. Dado que la variable `tenure` es `0` para todos estos casos, se trata de clientes completamente    nuevos que aún no han generado un cargo total (`TotalCharges`). Por lo tanto, se podría:
    * imputar con `0`, ya que son nuevos, o
    * imputar con el valor de `MonthlyCharges` para cada registro, asumiendo que ese será el primer       cargo que se les facture.

In [ ]:
pd.concat([
            df[["TotalCharges"]].describe().T
            , df[["TotalCharges"]].fillna(0).describe().T
            , df[["TotalCharges"]].fillna(df["MonthlyCharges"]).describe().T
        ])

**Decisión adoptada:** se reemplazan los valores nulos de `TotalCharges` por el valor de `MonthlyCharges`. Esto se debe a que los registros con valores nulos en `TotalCharges` corresponden exactamente a clientes con `tenure = 0` (clientes nuevos, sin cargos acumulados aún). Por lo tanto, usar el cargo mensual como estimación del cargo total es la opción más consistente con el contexto de negocio, sin sesgar la distribución de la variable con un valor arbitrario como el promedio.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"])

In [ ]:
print("items con valores nulos =", df.isnull().sum().sum())


## Técnicas para transformación de variables

### Numéricas
* **Escalamiento Min-Max**: útil cuando se requiere un rango común entre variables.
* **Estandarización (Z-score)**: útil para modelos sensibles a la escala (ej. Regresión Logística).
* **Transformación logarítmica**: útil cuando existen distribuciones altamente asimétricas.
* **Discretización (binning)**: convierte una variable continua en categorías interpretables.

Se identifican las variables numéricas del dataset.

In [ ]:
cols_num = list(df.select_dtypes(["int","float"]).columns)
df[cols_num].describe().T

Se evidencia dispersión entre las magnitudes de las variables numéricas, por lo que se recomienda normalizar o estandarizar los datos antes de aplicar cualquier modelo de machine learning.

**Nota importante:** el escalamiento en sí **no se ejecuta en este notebook**. Se deja documentado aquí como parte del diagnóstico, pero se aplica más adelante en `4_Modeling.ipynb`, y únicamente sobre el conjunto de `train` (para evitar fuga de información hacia el conjunto de `test`).

### Categóricas

* **Binarias**: True/False, Sí/No → 1 y 0.
* **One-Hot Encoding**: convertir cada categoría de la variable en una nueva columna con valores 1 y 0.
* **Label Encoding**: convierte cada categoría en un único valor numérico (útil para variables ordinales).

Se empieza por las variables dicotómicas, convertidas directamente a 1|0, por ejemplo: `No=0` y `Yes=1`.

In [ ]:
cols_to_bool = ["Partner","Dependents","PhoneService","PaperlessBilling","Churn"]
df[cols_to_bool] = df[cols_to_bool].apply(lambda x: x.map({"No":0, "Yes":1}))
df[cols_to_bool].describe().T

`gender` es una variable categórica, pero al tener solo 2 valores posibles se puede tratar de forma binaria igual que las anteriores.

In [ ]:
df["gender"] = df["gender"].map({"Male":0, "Female":1})
pd.concat([df["gender"].value_counts(), df["gender"].value_counts(normalize=True)], axis=1)

Se revisan las variables categóricas que tienen 3 opciones posibles (por ejemplo `Yes`, `No`, `No internet service`).

In [ ]:
print("Distribución de las varibales (en %):")
cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
for c in cols:
    print(c, (df[c].value_counts(normalize=True)*100).round(1).to_dict())

Para efectos de modelado, la categoría `No internet service` se trata como equivalente a `No`, ya que en la práctica ambas significan "el cliente no tiene ese servicio activo". Por lo tanto, estas variables se tratan como dicotómicas: `Yes=1` y cualquier otro valor (`No` o `No internet service`) = `0`.

In [ ]:
cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
for c in cols:
    filtro = df[c]=="Yes"
    df[c] = filtro.astype(int)

Se listan las variables categóricas que aún quedan pendientes por codificar — aquellas con más de 2 o 3 categorías sin un tratamiento binario claro, como `InternetService`, `Contract` o `PaymentMethod`.

In [ ]:
print("Variables pendientes por procesar (en %):")
cols_cat = list(df.select_dtypes(["object","str"]).columns)
for c in cols_cat:
    print(c, (df[c].value_counts(normalize=True)*100).round(1).to_dict())

### Conversión de variables a Dummies (One-Hot Encoding)

Para las variables categóricas restantes se aplica One-Hot Encoding: cada categoría se convierte en una columna binaria independiente. El `target` se conserva aparte y se reincorpora al final para no incluirlo por error en el proceso de codificación.

In [ ]:
df = pd.concat([
                df.drop(cols_cat + [target], axis=1),
                pd.get_dummies(df[cols_cat]).astype(int),
                df[target]
            ], axis=1
        )

In [ ]:
pd.concat([df.describe().T, pd.DataFrame(df.sum(), columns=["sum"])], axis=1)

# Análisis de correlación de variables

Con todas las variables ya en formato numérico (tras la codificación), se calcula la matriz de correlación completa para detectar redundancia entre variables antes de pasar al modelado.

In [ ]:
df_corr = df.corr()

La librería `seaborn` permite visualizar mejor la matriz de correlación que la tabla numérica cruda.

In [ ]:
## cargar librerias de visualizacion
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(12, 12))
sns.heatmap(
            df_corr
            , square=True
            , cmap="coolwarm"
            , annot=True, fmt=".0%"
            , annot_kws={"size": 7, 
                        #  "weight": "bold", 
                         "color": "black"}
            , cbar_kws={"shrink": 0.7}
            , vmin=-1, vmax=1
        )
plt.title("Correlación entre variables", fontsize=10
          )
plt.show()

Se ordenan las variables según su correlación directa con la variable objetivo (`Churn`), como primer filtro de relevancia.

In [ ]:
print(f"Variables más correlacionadas con el {target=}:")
df_corr.loc[target].sort_values(ascending=False)[1:]

Dado que la variable `TotalCharges` presenta una alta correlación con `tenure` y una correlación moderada con `MonthlyCharges`, además de haber requerido imputación por valores faltantes, se decide evaluar su eliminación para reducir posible redundancia y simplificar el conjunto de variables.

In [ ]:
df.drop("TotalCharges", axis=1, inplace=True, errors="ignore")

### Otras correlaciones

Más allá de la relación con el target, se revisan las correlaciones fuertes (|r| > 0.5) entre pares de variables predictoras, para identificar redundancia estructural (por ejemplo, generada por el propio One-Hot Encoding) frente a relaciones que sí aportan información de negocio.

In [ ]:
df_corr_otros = df_corr[
                            (df_corr.abs()!=1) 
                            & ((df_corr>0.5) | (df_corr<-0.5))
                    ].unstack().dropna().reset_index().rename(columns={0:"correlacion"})

df_corr_otros["relacion"] = df_corr_otros.iloc[:,:2].apply(lambda x: " <-> ".join(sorted(list(x))), axis=1)
df_corr_otros[["relacion", "correlacion"]].sort_values("correlacion", ascending=False).drop_duplicates().set_index("relacion")

Se evidencian algunas relaciones que no aportan explicación adicional en una revisión inicial, por ejemplo:

* `InternetService_No` ↔ `MonthlyCharges` (-0.763): relación negativa fuerte y esperada por la lógica del negocio. Los clientes sin servicio de internet tienden a presentar cargos mensuales considerablemente menores. Esta correlación surge principalmente de la codificación One-Hot de una categoría mutuamente excluyente y no aporta información adicional relevante sobre el comportamiento del cliente.
* `InternetService_DSL` ↔ `InternetService_Fiber optic` (-0.641): correlación negativa esperada, ya que ambas variables representan categorías mutuamente excluyentes de la variable original `InternetService`. Esta relación es consecuencia directa del proceso de One-Hot Encoding y no debe interpretarse como una relación estadística entre servicios, sino como una dependencia estructural de la codificación.
* `StreamingMovies` y `StreamingTV` presentan una correlación positiva moderada entre sí (r ≈ 0.53), lo que sugiere que ambos servicios suelen contratarse de manera conjunta.
* Tanto `StreamingMovies` como `StreamingTV` muestran correlaciones positivas moderadas con `MonthlyCharges` (r ≈ 0.63), indicando que los servicios de entretenimiento contribuyen al incremento del valor mensual facturado al cliente. Estas relaciones sí aportan información útil sobre el comportamiento y el valor comercial del cliente, por lo que las variables se conservan en el conjunto de características.

**Nota metodológica:** dentro del proceso de One-Hot Encoding también se puede usar `drop_first=True` para eliminar automáticamente una categoría de referencia por variable y reducir la multicolinealidad estructural descrita arriba.

Con base en el hallazgo anterior, se elimina `InternetService_No` por ser una columna redundante: su información ya queda representada por la ausencia de `1` en `InternetService_DSL` e `InternetService_Fiber optic` (categorías mutuamente excluyentes generadas por el mismo One-Hot Encoding). Conservarla no aporta información nueva y sí incrementa la colinealidad estructural del conjunto de variables.

In [ ]:
## eliminar catergoría "No internet service" de las variables categóricas relacionadas con el servicio de internet
df.drop("InternetService_No", axis=1, inplace=True)

# Resultados esperados

* Convertir variables al tipo de dato adecuado.
* Transformar variables numéricas para mejorar su representación.
* Codificar variables categóricas correctamente.
* Construir un conjunto de variables con sentido de negocio.
* Reducir redundancia entre variables.

Se estandarizan los nombres de columnas a minúsculas, por convención de estilo, antes de exportar el resultado final.

In [ ]:
df.columns = df.columns.str.lower()
df.head()

### Entrega final

Se exporta el dataset ya limpio, transformado y codificado a `../data/preprocessed_data.csv`. Este es el archivo que consume directamente `4_Modeling.ipynb` para entrenar y comparar los modelos de clasificación.

In [ ]:
df.to_csv("../data/preprocessed_data.csv")